# Actual pilot batch and resume preflight

Test actual CUDA FP32 width192/depth4/p4, microbatch8×accumulation16. K4 reference versus J2+2 recovery retains the full5000-step schedule. Uses pilot training data only; no held-out predictions. Expected runtime1–5minutes. Re-run creates a new isolated certificate. Checkpoints under checkpoints/<preflight_id>-<kind>-{reference,split}/. The certificate itself exercises resume in fresh model/optimizer state; a failure is retained.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
from fmm_jepa.tiny_pilot import pilot_preflight
from pathlib import Path
import datetime, uuid
ROOT=Path('/home/benjamin/workspace/ahr_go/fmm_jeppa')
preflight_id='pilot-preflight-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
artifact=ROOT/'artifacts/notebook_01'/preflight_id
result=pilot_preflight(ROOT,artifact)
print('CERTIFICATE:',artifact/'preflight_certificate.json')
print('STATUS:',result['status'])
print('Estimated sequential training hours:',result['estimated_pair_training_hours'])
print('Estimated with25percent overhead:',result['estimated_pair_with_overhead_hours'])